# Ablation Study for OTalign Gap Penalties

This notebook conducts an ablation study on the `otalign` model, specifically focusing on the contribution of different components in the dynamic gap penalty calculation from Unbalanced Optimal Transport (UOT) plans.

The study follows the plan outlined below:
1.  **Experiment 1: Fixed vs. Dynamic Gap Penalties:** Compare the full model against a version with traditional, fixed gap penalties.
2.  **Experiment 2: Gap Penalty Information Sources:** Analyze the contribution of marginal mass vs. dual potentials.
3.  **Experiment 3: Hyperparameter Sensitivity:** Test the sensitivity of key hyperparameters like `gamma`, `k_f`, and `score_scale`.

All experiments will be run on the SABmark-sup dataset using pre-computed transport plans.

> **For reproducible CLI execution**, use the standalone script:
> ```bash
> python scripts/run_ablation_study.py \
>   --dataset DeepFoldProtein/SABmark-dataset,sup,test \
>   --transport_plan_dir out/results/sabmark_sup/otalign_esm1b_lora_ft2_2/transport_plans
> ```
> Run `python scripts/run_ablation_study.py --help` for all available options.

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm


# Add project root to path to allow imports from scripts
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# OTAlign components
# Plotting

from otalign.align.uot_alignment import hard_alignment_from_transport
from otalign.quantize import dequantize
from scripts.dataset_utils import iter_pairs_from_dataset


pd.set_option("display.max_rows", 100)
print(f"PyTorch version: {torch.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Load Data and Configuration

Here, we load the SABmark dataset pairs and their pre-computed transport plans. We also need the ground-truth alignments to calculate F1 scores.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Configuration ---
# Define the dataset and the location of the pre-computed transport plans
DATASET_ID = "DeepFoldProtein/SABmark-dataset,sup,test"
TRANSPORT_PLAN_DIR = Path("out/results/sabmark_sup/otalign_esm1b_lora_ft2_2/transport_plans")

if not TRANSPORT_PLAN_DIR.exists():
    raise FileNotFoundError(f"Transport plan directory not found: {TRANSPORT_PLAN_DIR}\nPlease ensure you have run the benchmark and generated the plans first.")

# --- Load Data ---
dataset_iterator = iter_pairs_from_dataset(DATASET_ID)
all_data = []

print(f"Loading data from {TRANSPORT_PLAN_DIR}...")
for item in tqdm(list(dataset_iterator)):
    pair_id = item.get("pair_id", f"{item['seq1_id']}-{item['seq2_id']}")
    plan_path = TRANSPORT_PLAN_DIR / f"{pair_id}.npz"

    if plan_path.exists():
        try:
            data = np.load(plan_path, allow_pickle=True)
            # The quantized plan is stored in 'data', 'scale', 'zero_point'
            quantized_plan_data = {"data": data["data"], "scale": data["scale"], "zero_point": data["zero_point"]}

            all_data.append({"pair_id": pair_id, "P": dequantize(quantized_plan_data), "f": data["f"], "g": data["g"], "true_matches": {tuple(x) for x in item["ref_alignment"]}})
        except Exception as e:
            print(f"Warning: Could not load or process {plan_path}. Error: {e}")

print(f"Successfully loaded {len(all_data)} pairs with transport plans.")

## 3. Define Evaluation Metric (F1 Score)

In [ ]:
def calculate_f1(predicted_path, true_matches):
    """
    Calculates F1 score for a predicted alignment path.

    Args:
        predicted_path (list): List of tuples (q_idx, t_idx, op) from hard_alignment_from_transport.
        true_matches (set): Set of ground-truth matched pairs (q_idx, t_idx).
    """
    # Predicted matches are 1-based, convert to 0-based
    predicted_matches = {(q - 1, t - 1) for q, t, op in predicted_path if op == "M"}

    if not predicted_matches and not true_matches:
        return 1.0, 1.0, 1.0  # Precision, Recall, F1

    tp = len(predicted_matches.intersection(true_matches))
    fp = len(predicted_matches.difference(true_matches))
    fn = len(true_matches.difference(predicted_matches))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f1

## 4. Define Ablation Experiments

In [ ]:
def run_experiment_full_model(P, f, g, **kwargs):
    """Runs the standard OTalign model."""
    kwargs["mode"] = "glocal"
    return hard_alignment_from_transport(P, f=f, g=g, **kwargs)


def run_experiment_1_fixed_gaps(P, **kwargs):
    """Exp 1: Simulate fixed gap penalties by neutralizing dynamic components."""
    # By setting f=None, g=None, and gamma=0, we make the gap penalty calculation
    # independent of position-specific mass and duals, effectively using go_base and ge_base.
    kwargs["gamma"] = 0.0
    kwargs["mode"] = "glocal"
    return hard_alignment_from_transport(P, f=None, g=None, **kwargs)


def run_experiment_2a_mass_only(P, **kwargs):
    """Exp 2a: Use only marginal mass for gap penalties (f=None, g=None)."""
    kwargs["mode"] = "glocal"
    return hard_alignment_from_transport(P, f=None, g=None, **kwargs)


def run_experiment_2b_duals_only(P, f, g, **kwargs):
    """Exp 2b: Use only dual potentials for gap penalties (gamma=0)."""
    kwargs["gamma"] = 0.0
    kwargs["mode"] = "glocal"
    return hard_alignment_from_transport(P, f=f, g=g, **kwargs)


print("Experiment functions defined.")

## 5. Run Ablation Study

Now we loop through the dataset and run all defined experiments. We collect the F1 score for each configuration.

In [ ]:
results = []
default_params = {"go_base": 8.0, "ge_base": 1.0, "gamma": 1.0, "k_f": 0.75, "k_g": 0.75, "score_scale": 1.0}

for item in tqdm(all_data):
    P = item["P"]
    f = item["f"]
    g = item["g"]
    true_matches = item["true_matches"]
    pair_id = item["pair_id"]

    # --- Base Model ---
    aln = run_experiment_full_model(P, f, g, **default_params)
    _, _, f1 = calculate_f1(aln["path"], true_matches)
    results.append({"pair": pair_id, "config": "OTalign (Full Model)", "f1": f1})

    # --- Exp 1: Fixed Gaps ---
    aln = run_experiment_1_fixed_gaps(P, **default_params)
    _, _, f1 = calculate_f1(aln["path"], true_matches)
    results.append({"pair": pair_id, "config": "Exp 1: Fixed Gaps", "f1": f1})

    # --- Exp 2a: Mass Only ---
    aln = run_experiment_2a_mass_only(P, **default_params)
    _, _, f1 = calculate_f1(aln["path"], true_matches)
    results.append({"pair": pair_id, "config": "Exp 2a: Mass Only", "f1": f1})

    # --- Exp 2b: Duals Only ---
    aln = run_experiment_2b_duals_only(P, f, g, **default_params)
    _, _, f1 = calculate_f1(aln["path"], true_matches)
    results.append({"pair": pair_id, "config": "Exp 2b: Duals Only", "f1": f1})

    # --- Exp 3: Hyperparameter Sensitivity ---
    for gamma in [0.5, 2.0]:  # 1.0 is default
        params = default_params.copy()
        params["gamma"] = gamma
        aln = run_experiment_full_model(P, f, g, **params)
        _, _, f1 = calculate_f1(aln["path"], true_matches)
        results.append({"pair": pair_id, "config": f"Exp 3: gamma = {gamma}", "f1": f1})

    for k in [0.0, 1.5]:  # 0.75 is default
        params = default_params.copy()
        params["k_f"], params["k_g"] = k, k
        aln = run_experiment_full_model(P, f, g, **params)
        _, _, f1 = calculate_f1(aln["path"], true_matches)
        results.append({"pair": pair_id, "config": f"Exp 3: k_f = k_g = {k}", "f1": f1})

    for scale in [0.5, 2.0]:  # 1.0 is default
        params = default_params.copy()
        params["score_scale"] = scale
        aln = run_experiment_full_model(P, f, g, **params)
        _, _, f1 = calculate_f1(aln["path"], true_matches)
        results.append({"pair": pair_id, "config": f"Exp 3: score_scale = {scale}", "f1": f1})

results_df = pd.DataFrame(results)
print("Finished running experiments.")

## 6. Analyze Results

In [ ]:
summary = results_df.groupby("config")["f1"].mean().reset_index()

# Ensure the main model is present before calculating delta
if not summary[summary["config"] == "OTalign (Full Model)"].empty:
    base_f1 = summary[summary["config"] == "OTalign (Full Model)"]["f1"].iloc[0]
    summary["% Change"] = summary["f1"].apply(lambda f1: f"{(f1 - base_f1) / base_f1 * 100:.1f}%" if base_f1 > 0 else "N/A")
else:
    summary["% Change"] = "N/A"

summary.rename(columns={"config": "Configuration", "f1": "Mean F1 Score"}, inplace=True)

# Define a categorical type for sorting to match the proposal
config_order = [
    "OTalign (Full Model)",
    "Exp 1: Fixed Gaps",
    "Exp 2a: Mass Only",
    "Exp 2b: Duals Only",
    "Exp 3: gamma = 0.5",
    "Exp 3: gamma = 2.0",
    "Exp 3: k_f = k_g = 0.0",
    "Exp 3: k_f = k_g = 1.5",
    "Exp 3: score_scale = 0.5",
    "Exp 3: score_scale = 2.0",
]
summary["Configuration"] = pd.Categorical(summary["Configuration"], categories=config_order, ordered=True)
summary = summary.sort_values("Configuration")

summary["Mean F1 Score"] = summary["Mean F1 Score"].map("{:.3f}".format)

print("Ablation Study Results (SABmark-sup)")
display(summary.style.hide(axis="index"))